In [3]:
import pandas as pd
import numpy as np
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)


In [11]:
ball_by_ball_df = pd.read_csv(r"F:\CODING\ipl project\drive-download-20260415T150344Z-3-001\ball_by_ball_data.csv")

In [12]:
ball_by_ball_df.head()

,season_id,match_id,batter,bowler,non_striker,team_batting,team_bowling,over_number,ball_number,batter_runs,extras,total_runs,batsman_type,bowler_type,player_out,fielders_involved,is_wicket,is_wide_ball,is_no_ball,is_leg_bye,is_bye,is_penalty,wide_ball_runs,no_ball_runs,leg_bye_runs,bye_runs,penalty_runs,wicket_kind,is_super_over,innings
0,2008,335982,SC Ganguly,P Kumar,BB McCullum,Kolkata Knight Riders,Royal Challengers Bangalore,0,0,0,1,1,Left hand Bat,Right arm Medium,NaN,NaN,False,False,False,True,False,False,0,0,1,0,0,NaN,False,1
1,2008,335982,BB McCullum,P Kumar,SC Ganguly,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,0,0,0,Right hand Bat,Right arm Medium,NaN,NaN,False,False,False,False,False,False,0,0,0,0,0,NaN,False,1
2,2008,335982,BB McCullum,P Kumar,SC Ganguly,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,0,1,1,Right hand Bat,Right arm Medium,NaN,NaN,False,True,False,False,False,False,1,0,0,0,0,NaN,False,1
3,2008,335982,BB McCullum,P Kumar,SC Ganguly,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,0,0,0,Right hand Bat,Right arm Medium,NaN,NaN,False,False,False,False,False,False,0,0,0,0,0,NaN,False,1
4,2008,335982,BB McCullum,P Kumar,SC Ganguly,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,0,0,0,Right hand Bat,Right arm Medium,NaN,NaN,False,False,False,False,False,False,0,0,0,0,0,NaN,False,1


In [13]:
ball_by_ball_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 278205 entries, 0 to 278204
Data columns (total 30 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   season_id          278205 non-null  int64
 1   match_id           278205 non-null  int64
 2   batter             278205 non-null  str  
 3   bowler             278205 non-null  str  
 4   non_striker        278205 non-null  str  
 5   team_batting       278205 non-null  str  
 6   team_bowling       278205 non-null  str  
 7   over_number        278205 non-null  int64
 8   ball_number        278205 non-null  int64
 9   batter_runs        278205 non-null  int64
 10  extras             278205 non-null  int64
 11  total_runs         278205 non-null  int64
 12  batsman_type       278205 non-null  str  
 13  bowler_type        278205 non-null  str  
 14  player_out         13823 non-null   str  
 15  fielders_involved  13823 non-null   str  
 16  is_wicket          278205 non-null  bool 
 17  is

In [16]:
# -----------------------------------------------
# 1. Remove super over rows
# -----------------------------------------------
ball_by_ball_df = ball_by_ball_df[ball_by_ball_df['is_super_over'] == False]
ball_by_ball_df = ball_by_ball_df[ball_by_ball_df['innings'].isin([1, 2])]
print(f"After removing super overs: {ball_by_ball_df.shape}")
 
# -----------------------------------------------
# 2. Add phase column
# -----------------------------------------------
def get_phase(over):
    if over <= 5:
        return 'Powerplay'
    elif over <= 14:
        return 'Middle'
    else:
        return 'Death'
 
ball_by_ball_df['phase'] = ball_by_ball_df['over_number'].apply(get_phase)
 
# -----------------------------------------------
# 3. Add is_four and is_six
# -----------------------------------------------
ball_by_ball_df['is_four'] = (ball_by_ball_df['batter_runs'] == 4).astype(int)
ball_by_ball_df['is_six'] = (ball_by_ball_df['batter_runs'] == 6).astype(int)
 
# -----------------------------------------------
# 4. Convert bool columns to int (MySQL friendly)
# -----------------------------------------------
ball_by_ball_df['is_wicket'] = ball_by_ball_df['is_wicket'].astype(int)
ball_by_ball_df['is_wide_ball'] = ball_by_ball_df['is_wide_ball'].astype(int)
ball_by_ball_df['is_no_ball'] = ball_by_ball_df['is_no_ball'].astype(int)
 
# -----------------------------------------------
# 5. Rename season_id to season
# -----------------------------------------------
ball_by_ball_df.rename(columns={'season_id': 'season'}, inplace=True)
 
# -----------------------------------------------
# 6. Keep only required columns
# -----------------------------------------------
keep_cols = [
    'season', 'match_id', 'batter', 'bowler',
    'team_batting', 'team_bowling',
    'over_number', 'innings', 'phase',
    'batter_runs', 'total_runs',
    'is_wicket', 'wicket_kind',
    'is_four', 'is_six',
    'is_wide_ball', 'is_no_ball'
]
 
ball_by_ball_df = ball_by_ball_df[keep_cols]
print(f"After column selection: {ball_by_ball_df.shape}")
 
# -----------------------------------------------
# 7. Final check
# -----------------------------------------------
print(f"\nColumns: {ball_by_ball_df.columns.tolist()}")
print(f"\nNull counts:\n{ball_by_ball_df.isnull().sum()}")
print(f"\nPhase distribution:\n{ball_by_ball_df['phase'].value_counts()}")
print(f"\nSample:\n{ball_by_ball_df.head(3)}")
 
# -----------------------------------------------
# 8. Export
# -----------------------------------------------
ball_by_ball_df.to_csv('ball_by_ball_clean.csv', index=False)
print("\n✅ Exported: ball_by_ball_clean.csv")

After removing super overs: (278034, 30)
After column selection: (278034, 17)

Columns: ['season', 'match_id', 'batter', 'bowler', 'team_batting', 'team_bowling', 'over_number', 'innings', 'phase', 'batter_runs', 'total_runs', 'is_wicket', 'wicket_kind', 'is_four', 'is_six', 'is_wide_ball', 'is_no_ball']

Null counts:
season               0
match_id             0
batter               0
bowler               0
team_batting         0
team_bowling         0
over_number          0
innings              0
phase                0
batter_runs          0
total_runs           0
is_wicket            0
wicket_kind     264240
is_four              0
is_six               0
is_wide_ball         0
is_no_ball           0
dtype: int64

Phase distribution:
phase
Middle       127548
Powerplay     87198
Death         63288
Name: count, dtype: int64

Sample:
   season  match_id       batter   bowler           team_batting  \
0    2008    335982   SC Ganguly  P Kumar  Kolkata Knight Riders   
1    2008    33598